# Large Data Efficiency Notebook

## Business Context

Cílem notebooku je ukázat, jak efektivně pracovat s většími datasety bez automatického použití distribuovaných nástrojů.

Notebook porovnává různé způsoby načítání a zpracování dat a ukazuje, jak lze snížit množství dat přenášených do analytického prostředí.

## Setup and Paths

V této části importujeme potřebné knihovny a nastavíme cesty. Větší testovací soubory budou vytvořeny automaticky a nebudou ukládány do Git repozitáře.

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parents[1]

GENERATED_DATA_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "generated"
)

CSV_PATH = (
    GENERATED_DATA_DIR
    / "large_sales.csv"
)

PARQUET_PATH = (
    GENERATED_DATA_DIR
    / "large_sales.parquet"
)

DATABASE_PATH = (
    GENERATED_DATA_DIR
    / "large_sales.db"
)


GENERATED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Projektová složka:", PROJECT_ROOT.name)
print("Pracovní datová složka:", GENERATED_DATA_DIR)

print("CSV soubor:", CSV_PATH.name)
print("Parquet soubor:", PARQUET_PATH.name)
print("SQLite databáze:", DATABASE_PATH.name)

Projektová složka: data-analytics-tools-portfolio
Pracovní datová složka: c:\Users\frant\Documents\data-analytics-workspace\data-analytics-tools-portfolio\datasets\generated
CSV soubor: large_sales.csv
Parquet soubor: large_sales.parquet
SQLite databáze: large_sales.db


## Test Dataset Generation

Pro praktické porovnání vytvoříme reprodukovatelný syntetický dataset prodejních transakcí. Dataset bude obsahovat 300 000 řádků a následně ho uložíme ve více formátech.

Syntetická data slouží pouze k demonstraci práce s větším objemem dat.

In [2]:
ROW_COUNT = 300_000

np.random.seed(42)

date_range = pd.date_range(
    start="2024-01-01",
    end="2025-12-31",
    freq="D"
)

products = [
    "Laptop",
    "Monitor",
    "Keyboard",
    "Mouse",
    "Headphones"
]

regions = [
    "Praha",
    "Plzeň",
    "Brno",
    "Ostrava",
    "Olomouc"
]

channels = [
    "Online",
    "Store",
    "Partner"
]

product_prices = {
    "Laptop": 25000,
    "Monitor": 7000,
    "Keyboard": 1500,
    "Mouse": 800,
    "Headphones": 2500
}

In [3]:
df_generated = pd.DataFrame({
    "order_id": np.arange(
        1,
        ROW_COUNT + 1
    ),
    "order_date": np.random.choice(
        date_range,
        size=ROW_COUNT
    )
})

print("Rozměry datasetu:", df_generated.shape)

display(df_generated.head())

Rozměry datasetu: (300000, 2)


,order_id,order_date
0,1,2024-04-12
1,2,2025-03-11
2,3,2024-09-27
3,4,2024-04-16
4,5,2024-03-12


In [4]:
df_generated["customer_id"] = np.random.randint(
    1000,
    50000,
    size=ROW_COUNT
)

df_generated["region"] = np.random.choice(
    regions,
    size=ROW_COUNT
)

df_generated["product"] = np.random.choice(
    products,
    size=ROW_COUNT
)

df_generated["channel"] = np.random.choice(
    channels,
    size=ROW_COUNT
)

print("Rozměry datasetu:", df_generated.shape)

df_generated.head()

Rozměry datasetu: (300000, 6)


,order_id,order_date,customer_id,region,product,channel
0,1,2024-04-12,7990,Brno,Headphones,Partner
1,2,2025-03-11,8490,Plzeň,Monitor,Online
2,3,2024-09-27,31362,Olomouc,Laptop,Online
3,4,2024-04-16,10859,Praha,Headphones,Online
4,5,2024-03-12,7995,Praha,Mouse,Partner


In [5]:
df_generated["quantity"] = np.random.randint(
    1,
    6,
    size=ROW_COUNT
)

df_generated["discount_pct"] = np.random.choice(
    [0, 0.05, 0.10, 0.15],
    size=ROW_COUNT
)

print("Rozměry datasetu:", df_generated.shape)

df_generated.head()

Rozměry datasetu: (300000, 8)


,order_id,order_date,customer_id,region,product,channel,quantity,discount_pct
0,1,2024-04-12,7990,Brno,Headphones,Partner,3,0.15
1,2,2025-03-11,8490,Plzeň,Monitor,Online,3,0.00
2,3,2024-09-27,31362,Olomouc,Laptop,Online,2,0.15
3,4,2024-04-16,10859,Praha,Headphones,Online,5,0.10
4,5,2024-03-12,7995,Praha,Mouse,Partner,2,0.05


In [6]:
df_generated["unit_price"] = (
    df_generated["product"]
    .map(product_prices)
)

df_generated["revenue"] = (
    df_generated["quantity"]
    * df_generated["unit_price"]
    * (1 - df_generated["discount_pct"])
).round(2)

print("Rozměry datasetu:", df_generated.shape)

df_generated[
    [
        "product",
        "quantity",
        "unit_price",
        "discount_pct",
        "revenue"
    ]
].head()

Rozměry datasetu: (300000, 10)


,product,quantity,unit_price,discount_pct,revenue
0,Headphones,3,2500,0.15,6375.0
1,Monitor,3,7000,0.00,21000.0
2,Laptop,2,25000,0.15,42500.0
3,Headphones,5,2500,0.10,11250.0
4,Mouse,2,800,0.05,1520.0


## Initial Data Inspection

Před uložením datasetu zkontrolujeme jeho rozměry, datové typy a přibližnou spotřebu operační paměti.

In [7]:
print("Počet řádků:", df_generated.shape[0])
print("Počet sloupců:", df_generated.shape[1])

df_generated.info()

Počet řádků: 300000
Počet sloupců: 10
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      300000 non-null  int64         
 1   order_date    300000 non-null  datetime64[us]
 2   customer_id   300000 non-null  int32         
 3   region        300000 non-null  str           
 4   product       300000 non-null  str           
 5   channel       300000 non-null  str           
 6   quantity      300000 non-null  int32         
 7   discount_pct  300000 non-null  float64       
 8   unit_price    300000 non-null  int64         
 9   revenue       300000 non-null  float64       
dtypes: datetime64[us](1), float64(2), int32(2), int64(2), str(3)
memory usage: 26.0 MB


## CSV Disk Size

Dataset uložíme jako CSV a porovnáme velikost souboru na disku se spotřebou DataFrame v operační paměti.

In [8]:
df_generated.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8"
)

csv_size_bytes = CSV_PATH.stat().st_size

csv_size_mb = (
    csv_size_bytes
    / 1024 ** 2
)

memory_size_bytes = (
    df_generated
    .memory_usage(deep=True)
    .sum()
)

memory_size_mb = (
    memory_size_bytes
    / 1024 ** 2
)

print(
    "Velikost CSV na disku:",
    round(csv_size_mb, 2),
    "MB"
)

print(
    "Velikost DataFrame v paměti:",
    round(memory_size_mb, 2),
    "MB"
)

Velikost CSV na disku: 18.7 MB
Velikost DataFrame v paměti: 26.04 MB


In [9]:
df_generated.info(
    memory_usage="deep"
)

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      300000 non-null  int64         
 1   order_date    300000 non-null  datetime64[us]
 2   customer_id   300000 non-null  int32         
 3   region        300000 non-null  str           
 4   product       300000 non-null  str           
 5   channel       300000 non-null  str           
 6   quantity      300000 non-null  int32         
 7   discount_pct  300000 non-null  float64       
 8   unit_price    300000 non-null  int64         
 9   revenue       300000 non-null  float64       
dtypes: datetime64[us](1), float64(2), int32(2), int64(2), str(3)
memory usage: 26.0 MB


### Memory Comparison

CSV soubor zabírá na disku 18,70 MB, zatímco načtený DataFrame zabírá při započtení textových hodnot 62,42 MB. Data po načtení tedy vyžadují přibližně 3,3krát více operační paměti než samotný soubor.

## Memory Usage by Column

Spotřebu paměti rozdělíme podle jednotlivých sloupců. Cílem je zjistit, které části DataFrame mají největší vliv na celkovou spotřebu paměti. Datové typy zatím neměníme.

In [10]:
memory_by_column_mb = (
    df_generated
    .memory_usage(deep=True)
    / 1024 ** 2
)

memory_by_column_mb = (
    memory_by_column_mb
    .sort_values(ascending=False)
)

memory_by_column_mb

product         4.349045
channel         4.005004
region          3.948115
revenue         2.288818
order_date      2.288818
order_id        2.288818
discount_pct    2.288818
unit_price      2.288818
customer_id     1.144409
quantity        1.144409
Index           0.000126
dtype: float64

## CSV vs. Parquet

CSV je textový formát, který ukládá celé záznamy po řádcích. Je široce podporovaný a snadno čitelný, ale neuchovává spolehlivě datové typy a při analytickém čtení musí být text znovu zpracován.

Parquet je binární sloupcový formát určený pro analytickou práci. Uchovává informace o datových typech, podporuje kompresi a umožňuje efektivně načítat pouze potřebné sloupce.

Parquet je vhodný zejména pro opakované analytické zpracování větších datasetů. CSV zůstává praktické pro jednoduchou výměnu dat mezi různými nástroji.

### Parquet Export and Size Comparison

Stejný DataFrame uložíme jako Parquet a porovnáme velikost výsledného souboru s CSV. Oba soubory budou obsahovat stejných 300 000 záznamů a stejné sloupce.

In [11]:
df_generated.to_parquet(
    PARQUET_PATH,
    index=False
)

parquet_size_bytes = (
    PARQUET_PATH
    .stat()
    .st_size
)

parquet_size_mb = (
    parquet_size_bytes
    / 1024 ** 2
)

print(
    "Velikost CSV:",
    round(csv_size_mb, 2),
    "MB"
)

print(
    "Velikost Parquet:",
    round(parquet_size_mb, 2),
    "MB"
)

print(
    "Parquet soubor existuje:",
    PARQUET_PATH.exists()
)

size_reduction_pct = (
    1
    - parquet_size_mb / csv_size_mb
) * 100

print(
    "Úspora místa:",
    round(size_reduction_pct, 2),
    "%"
)

Velikost CSV: 18.7 MB
Velikost Parquet: 3.36 MB
Parquet soubor existuje: True
Úspora místa: 82.03 %


## SQLite Storage

Stejný dataset uložíme do SQLite databáze. Na rozdíl od CSV a Parquetu umožňuje databáze filtrovat a agregovat data pomocí SQL ještě před jejich načtením do pandas nebo jiného analytického nástroje.

In [12]:
connection = sqlite3.connect(
    DATABASE_PATH
)

df_generated.to_sql(
    "sales",
    connection,
    if_exists="replace",
    index=False
)

connection.close()

print(
    "SQLite databáze existuje:",
    DATABASE_PATH.exists()
)

SQLite databáze existuje: True


In [13]:
connection = sqlite3.connect(DATABASE_PATH)

query = """
SELECT *
FROM sales
"""

df_sql = pd.read_sql(
    query,
    connection,
    parse_dates=["order_date"]
)

connection.close()

print("Počet řádků:", df_sql.shape[0])
print("Počet sloupců:", df_sql.shape[1])

display(df_sql.head())

Počet řádků: 300000
Počet sloupců: 10


,order_id,order_date,customer_id,region,product,channel,quantity,discount_pct,unit_price,revenue
0,1,2024-04-12,7990,Brno,Headphones,Partner,3,0.15,2500,6375.0
1,2,2025-03-11,8490,Plzeň,Monitor,Online,3,0.00,7000,21000.0
2,3,2024-09-27,31362,Olomouc,Laptop,Online,2,0.15,25000,42500.0
3,4,2024-04-16,10859,Praha,Headphones,Online,5,0.10,2500,11250.0
4,5,2024-03-12,7995,Praha,Mouse,Partner,2,0.05,800,1520.0


## Porovnání velikosti souborů

Porovnáme velikost stejného datasetu uloženého ve formátech CSV, Parquet a SQLite. Všechny soubory obsahují stejných 300 000 řádků a 10 sloupců.

In [14]:
csv_size_mb = (
    CSV_PATH.stat().st_size
    / 1024 ** 2
)

parquet_size_mb = (
    PARQUET_PATH.stat().st_size
    / 1024 ** 2
)

sqlite_size_mb = (
    DATABASE_PATH.stat().st_size
    / 1024 ** 2
)

print("Velikost CSV:", round(csv_size_mb, 2), "MB")
print("Velikost Parquet:", round(parquet_size_mb, 2), "MB")
print("Velikost SQLite:", round(sqlite_size_mb, 2), "MB")

Velikost CSV: 18.7 MB
Velikost Parquet: 3.36 MB
Velikost SQLite: 20.64 MB


### Zjištění

Parquet byl s velikostí 3,36 MB přibližně 5,6krát menší než CSV. Díky sloupcovému uložení a kompresi byl nejúspornějším z porovnávaných formátů.

SQLite databáze měla velikost 20,64 MB a byla mírně větší než CSV. Databáze kromě samotných hodnot ukládá také schéma a interní struktury potřebné pro práci s daty. Její hlavní výhodou proto není minimální velikost souboru, ale možnost filtrovat, agregovat a spojovat data pomocí SQL.

## Loading Speed Comparison

Porovnáme dobu potřebnou pro načtení celého datasetu z CSV, Parquetu a SQLite. Každý způsob načte stejných 300 000 řádků a 10 sloupců.

Naměřené hodnoty jsou orientační, protože rychlost může ovlivnit výkon počítače a mezipaměť operačního systému.

In [15]:
import time

start_time = time.perf_counter()

df_csv_loaded = pd.read_csv(
    CSV_PATH,
    parse_dates=["order_date"]
)

csv_load_time = (
    time.perf_counter()
    - start_time
)


start_time = time.perf_counter()

df_parquet_loaded = pd.read_parquet(
    PARQUET_PATH
)

parquet_load_time = (
    time.perf_counter()
    - start_time
)


start_time = time.perf_counter()

connection = sqlite3.connect(DATABASE_PATH)

df_sql_loaded = pd.read_sql(
    """
    SELECT *
    FROM sales
    """,
    connection,
    parse_dates=["order_date"]
)

connection.close()

sqlite_load_time = (
    time.perf_counter()
    - start_time
)


print(
    "Načtení CSV:",
    round(csv_load_time, 3),
    "sekund"
)

print(
    "Načtení Parquet:",
    round(parquet_load_time, 3),
    "sekund"
)

print(
    "Načtení SQLite:",
    round(sqlite_load_time, 3),
    "sekund"
)

Načtení CSV: 0.419 sekund
Načtení Parquet: 0.067 sekund
Načtení SQLite: 1.142 sekund


### Zjištění

První načtení Parquetu trvalo výrazně déle kvůli počáteční inicializaci knihovny PyArrow. Při opakovaném načtení byl Parquet nejrychlejší.

CSV musí při každém načtení zpracovat text a převést jej na datové typy. Parquet ukládá data ve sloupcovém binárním formátu a uchovává informace o datových typech.

Načtení celé tabulky ze SQLite bylo pomalejší než načtení CSV a Parquetu. Výhoda databáze se projeví především tehdy, když SQL dotaz omezí nebo agreguje data ještě před jejich načtením do Pandas.

## Memory Usage After Loading

Po načtení se CSV, Parquet i SQLite převedou na Pandas DataFrame. Porovnáme jejich spotřebu operační paměti a ověříme, zda způsob uložení ovlivnil výsledné datové typy.

In [16]:
csv_memory_mb = (
    df_csv_loaded
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

parquet_memory_mb = (
    df_parquet_loaded
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

sqlite_memory_mb = (
    df_sql_loaded
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "CSV DataFrame:",
    round(csv_memory_mb, 2),
    "MB"
)

print(
    "Parquet DataFrame:",
    round(parquet_memory_mb, 2),
    "MB"
)

print(
    "SQLite DataFrame:",
    round(sqlite_memory_mb, 2),
    "MB"
)

CSV DataFrame: 28.32 MB
Parquet DataFrame: 26.04 MB
SQLite DataFrame: 28.32 MB


In [18]:
dtype_comparison = pd.DataFrame({
    "CSV": df_csv_loaded.dtypes.astype(str),
    "Parquet": df_parquet_loaded.dtypes.astype(str),
    "SQLite": df_sql_loaded.dtypes.astype(str)
})

display(dtype_comparison)

,CSV,Parquet,SQLite
order_id,int64,int64,int64
order_date,datetime64[us],datetime64[us],datetime64[us]
customer_id,int64,int32,int64
region,str,str,str
product,str,str,str
channel,str,str,str
quantity,int64,int32,int64
discount_pct,float64,float64,float64
unit_price,int64,int64,int64
revenue,float64,float64,float64


### Zjištění

Po načtení zabíraly DataFrame z CSV a SQLite 28,32 MB, zatímco DataFrame z Parquetu 26,04 MB.

Parquet zachoval původní datové typy `int32` u sloupců `customer_id` a `quantity`. CSV datové typy neuchovává a SQLite nerozlišuje šířku celočíselného typu, proto Pandas oba sloupce načetl jako `int64`.

Rozdíl přibližně 2,28 MB odpovídá vyšší paměťové náročnosti dvou sloupců převedených z `int32` na `int64`.

## Loading Selected Columns

Při analytické práci často nepotřebujeme všechny sloupce zdrojového datasetu. Načtením pouze potřebných sloupců můžeme snížit spotřebu operační paměti a někdy také zrychlit načítání.

U CSV se celý soubor stále musí projít, protože jsou data uložena po řádcích. Do výsledného DataFrame se však uloží pouze vybrané sloupce.

Parquet ukládá data po sloupcích, a proto může ze souboru fyzicky načíst jen požadované sloupce. V SQL můžeme potřebné sloupce určit přímo pomocí příkazu `SELECT`.

In [23]:
selected_columns = [
    "order_date",
    "region",
    "product",
    "revenue",
    "quantity"
]

In [24]:
start_time = time.perf_counter()

df_csv_selected = pd.read_csv(
    CSV_PATH,
    usecols=selected_columns,
    parse_dates=["order_date"]
)

csv_selected_time = (
    time.perf_counter()
    - start_time
)


start_time = time.perf_counter()

df_parquet_selected = pd.read_parquet(
    PARQUET_PATH,
    columns=selected_columns
)

parquet_selected_time = (
    time.perf_counter()
    - start_time
)


start_time = time.perf_counter()

connection = sqlite3.connect(DATABASE_PATH)

df_sql_selected = pd.read_sql(
    """
    SELECT
        order_date,
        region,
        product,
        revenue,
        quantity
    FROM sales
    """,
    connection,
    parse_dates=["order_date"]
)

connection.close()

sql_selected_time = (
    time.perf_counter()
    - start_time
)


print(
    "CSV – vybrané sloupce:",
    round(csv_selected_time, 3),
    "sekund"
)

print(
    "Parquet – vybrané sloupce:",
    round(parquet_selected_time, 3),
    "sekund"
)

print(
    "SQLite – vybrané sloupce:",
    round(sql_selected_time, 3),
    "sekund"
)

CSV – vybrané sloupce: 0.489 sekund
Parquet – vybrané sloupce: 0.021 sekund
SQLite – vybrané sloupce: 0.694 sekund


In [25]:
print("CSV shape:", df_csv_selected.shape)
print("Parquet shape:", df_parquet_selected.shape)
print("SQLite shape:", df_sql_selected.shape)

CSV shape: (300000, 5)
Parquet shape: (300000, 5)
SQLite shape: (300000, 5)


In [26]:
csv_selected_memory_mb = (
    df_csv_selected
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

parquet_selected_memory_mb = (
    df_parquet_selected
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

sql_selected_memory_mb = (
    df_sql_selected
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "CSV – paměť:",
    round(csv_selected_memory_mb, 2),
    "MB"
)

print(
    "Parquet – paměť:",
    round(parquet_selected_memory_mb, 2),
    "MB"
)

print(
    "SQLite – paměť:",
    round(sql_selected_memory_mb, 2),
    "MB"
)

CSV – paměť: 15.16 MB
Parquet – paměť: 14.02 MB
SQLite – paměť: 15.16 MB


### Zjištění

Načtením pěti vybraných sloupců se výrazně snížila spotřeba paměti oproti načtení celého datasetu.

CSV a SQLite DataFrame zabíraly 15,16 MB, zatímco Parquet DataFrame zabíral 14,02 MB. Rozdíl 1,14 MB způsobil sloupec `quantity`. Parquet zachoval jeho původní typ `int32`, zatímco CSV a SQLite jej načetly jako `int64`.

Parquet byl také nejrychlejší, protože díky sloupcovému uložení mohl fyzicky přečíst pouze požadované sloupce. CSV muselo projít celý řádkový soubor a SQLite vrátilo vybrané sloupce prostřednictvím SQL dotazu.

## Filtering at the Source and SQL Pushdown

Cílem je získat pouze pět vybraných sloupců a řádky z roku 2025.

SQL pushdown znamená, že výběr sloupců a filtrování řádků provede databáze ještě před načtením dat do Pandas. Python proto obdrží pouze výsledná data, nikoliv celou zdrojovou tabulku.

In [27]:
start_time = time.perf_counter()

connection = sqlite3.connect(DATABASE_PATH)

df_sql_2025 = pd.read_sql(
    """
    SELECT
        order_date,
        region,
        product,
        quantity,
        revenue
    FROM sales
    WHERE order_date BETWEEN
        '2025-01-01'
        AND '2025-12-31 23:59:59'
    """,
    connection,
    parse_dates=["order_date"]
)

connection.close()

sql_filter_time = (
    time.perf_counter()
    - start_time
)

sql_filter_memory_mb = (
    df_sql_2025
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "Čas načtení:",
    round(sql_filter_time, 3),
    "sekund"
)

print("Shape:", df_sql_2025.shape)

print(
    "Paměť:",
    round(sql_filter_memory_mb, 2),
    "MB"
)

display(df_sql_2025.head())

Čas načtení: 0.467 sekund
Shape: (149589, 5)
Paměť: 7.56 MB


,order_date,region,product,quantity,revenue
0,2025-03-11,Plzeň,Monitor,3,21000.0
1,2025-12-01,Olomouc,Headphones,3,7500.0
2,2025-09-06,Plzeň,Laptop,2,50000.0
3,2025-04-11,Olomouc,Mouse,1,800.0
4,2025-04-03,Olomouc,Keyboard,3,4275.0


In [29]:
start_time = time.perf_counter()

df_csv_source = pd.read_csv(
    CSV_PATH,
    usecols=selected_columns,
    parse_dates=["order_date"]
)

df_csv_2025 = df_csv_source[
    df_csv_source["order_date"].between(
        "2025-01-01",
        "2025-12-31 23:59:59"
    )
].copy()

df_csv_2025 = df_csv_2025.reset_index(
    drop=True
)

csv_filter_time = (
    time.perf_counter()
    - start_time
)

csv_source_memory_mb = (
    df_csv_source
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

csv_filter_memory_mb = (
    df_csv_2025
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "Čas načtení a filtrování:",
    round(csv_filter_time, 3),
    "sekund"
)

print("Načtený zdroj:", df_csv_source.shape)
print("Filtrovaný výsledek:", df_csv_2025.shape)

print(
    "Paměť načteného zdroje:",
    round(csv_source_memory_mb, 2),
    "MB"
)

print(
    "Paměť výsledku:",
    round(csv_filter_memory_mb, 2),
    "MB"
)

Čas načtení a filtrování: 0.404 sekund
Načtený zdroj: (300000, 5)
Filtrovaný výsledek: (149589, 5)
Paměť načteného zdroje: 15.16 MB
Paměť výsledku: 7.6 MB


In [30]:
start_time = time.perf_counter()

df_parquet_2025 = pd.read_parquet(
    PARQUET_PATH,
    columns=selected_columns,
    filters=[
        (
            "order_date",
            ">=",
            pd.Timestamp("2025-01-01")
        ),
        (
            "order_date",
            "<",
            pd.Timestamp("2026-01-01")
        )
    ]
)

df_parquet_2025 = df_parquet_2025.reset_index(
    drop=True
)

parquet_filter_time = (
    time.perf_counter()
    - start_time
)

parquet_filter_memory_mb = (
    df_parquet_2025
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "Čas načtení a filtrování:",
    round(parquet_filter_time, 3),
    "sekund"
)

print("Filtrovaný výsledek:", df_parquet_2025.shape)

print(
    "Paměť výsledku:",
    round(parquet_filter_memory_mb, 2),
    "MB"
)

Čas načtení a filtrování: 0.028 sekund
Filtrovaný výsledek: (149589, 5)
Paměť výsledku: 7.03 MB


### Zjištění

Všechny tři postupy vrátily stejných 149 589 řádků z roku 2025.

Při práci s CSV načetl Pandas nejprve všech 300 000 řádků do zdrojového DataFrame o velikosti 15,16 MB. Následně Python pomocí Pandas vyfiltroval řádky z roku 2025.

Při práci se SQLite provedla filtrování databáze a do Pandas předala pouze výsledných 149 589 řádků. Tím se omezilo množství dat načtených do Pythonu.

Při práci s Parquetem provedl Parquet engine PyArrow filtrování během načítání a do výsledného DataFrame předal pouze požadované řádky. Parquet zároveň zachoval sloupec `quantity` jako `int32`, takže výsledný DataFrame zabíral méně paměti.

Naměřené časy jsou orientační a platí pro tento konkrétní lokální dataset a prostředí.

## Aggregation Before Loading into Power BI

Power BI nemusí vždy načítat všechny detailní řádky. Pokud report potřebuje pouze předem definovaný souhrn, může agregaci provést již databáze.

V tomto scénáři připravíme měsíční tržby a počet objednávek podle regionu. SQLite provede agregaci pomocí SQL a do Pandas se načte pouze výsledná souhrnná tabulka.

Tím lze výrazně snížit počet řádků, spotřebu paměti a množství dat načítaných do Power BI.


In [31]:
start_time = time.perf_counter()

connection = sqlite3.connect(DATABASE_PATH)

df_power_bi_summary = pd.read_sql(
    """
    SELECT
        strftime('%Y-%m', order_date) AS year_month,
        region,
        SUM(revenue) AS total_revenue,
        COUNT(*) AS order_count
    FROM sales
    GROUP BY
        strftime('%Y-%m', order_date),
        region
    ORDER BY
        year_month,
        region
    """,
    connection
)

connection.close()

aggregation_time = (
    time.perf_counter()
    - start_time
)

summary_memory_mb = (
    df_power_bi_summary
    .memory_usage(deep=True)
    .sum()
    / 1024 ** 2
)

print(
    "Čas SQL agregace a načtení:",
    round(aggregation_time, 3),
    "sekund"
)

print("Shape:", df_power_bi_summary.shape)

print(
    "Paměť výsledku:",
    round(summary_memory_mb, 4),
    "MB"
)

display(df_power_bi_summary.head(10))

Čas SQL agregace a načtení: 1.433 sekund
Shape: (120, 4)
Paměť výsledku: 0.0053 MB


,year_month,region,total_revenue,order_count
0,2024-01,Brno,53828615.0,2637
1,2024-01,Olomouc,51539515.0,2556
2,2024-01,Ostrava,50905260.0,2474
3,2024-01,Plzeň,50868990.0,2502
4,2024-01,Praha,51626050.0,2548
5,2024-02,Brno,47510810.0,2372
6,2024-02,Olomouc,48397120.0,2321
7,2024-02,Ostrava,47753220.0,2412
8,2024-02,Plzeň,49405935.0,2380
9,2024-02,Praha,47227670.0,2328


### Zjištění

SQLite agregovalo 300 000 detailních řádků do 120 řádků představujících kombinace měsíce a regionu. Počet řádků se tak snížil 2 500krát a výsledný DataFrame zabíral pouze přibližně 0,0053 MB.

Agregační SQL dotaz trval déle než jednoduché načtení tabulky, protože databáze musela vytvořit měsíční skupiny a vypočítat součty a počty záznamů.

Hlavní přínos předběžné agregace proto není vždy kratší doba samotného SQL dotazu. Přínosem je především výrazné omezení množství dat přenášených do Pandas nebo Power BI a menší výsledný datový model.

Agregovaný výstup je vhodný pouze tehdy, pokud report nepotřebuje detail jednotlivých objednávek.

## Processing CSV in Chunks

Pokud je CSV soubor příliš velký pro načtení do operační paměti najednou, může Pandas soubor zpracovávat po menších částech.

Parametr `chunksize` určuje počet řádků načtených v jednom kroku. Jednotlivým částem se říká chunks.

V tomto scénáři rozdělíme dataset s 300 000 řádky na části po 50 000 řádcích. V paměti tak bude postupně vždy pouze jedna část datasetu.

Nad jednotlivými částmi vypočítáme tržby podle regionu a dílčí výsledky následně spojíme do celkového souhrnu.

In [33]:
start_time = time.perf_counter()

csv_chunks = pd.read_csv(
    CSV_PATH,
    usecols=[
        "region",
        "revenue"
    ],
    chunksize=50000
)

chunk_summaries = []
chunk_count = 0

for chunk in csv_chunks:
    chunk_summary = (
        chunk
        .groupby(
            "region",
            as_index=False
        )["revenue"]
        .sum()
    )

    chunk_summaries.append(
        chunk_summary
    )

    chunk_count = chunk_count + 1


regional_revenue_chunks = (
    pd.concat(
        chunk_summaries,
        ignore_index=True
    )
    .groupby(
        "region",
        as_index=False
    )["revenue"]
    .sum()
    .sort_values(
        by="revenue",
        ascending=False,
        ignore_index=True
    )
)

chunks_processing_time = (
    time.perf_counter()
    - start_time
)

print("Počet zpracovaných chunks:", chunk_count)

print(
    "Čas zpracování:",
    round(chunks_processing_time, 3),
    "sekund"
)

display(
    regional_revenue_chunks.style.format({
        "revenue": "{:,.2f}"
    })
)

Počet zpracovaných chunks: 6
Čas zpracování: 0.561 sekund


,region,revenue
0,Brno,"1,234,397,285.00"
1,Praha,"1,231,990,720.00"
2,Plzeň,"1,225,560,370.00"
3,Olomouc,"1,221,459,670.00"
4,Ostrava,"1,211,618,150.00"


In [34]:
regional_revenue_full = (
    df_csv_loaded
    .groupby(
        "region",
        as_index=False
    )["revenue"]
    .sum()
    .sort_values(
        by="revenue",
        ascending=False,
        ignore_index=True
    )
)

same_result = np.allclose(
    regional_revenue_chunks["revenue"],
    regional_revenue_full["revenue"]
)

print(
    "Výsledek chunks odpovídá celému DataFrame:",
    same_result
)

Výsledek chunks odpovídá celému DataFrame: True


### Zjištění

CSV soubor s 300 000 řádky byl zpracován v šesti částech po 50 000 řádcích.

V každém kroku byl v paměti pouze jeden chunk. Nad každou částí vznikl malý souhrn tržeb podle regionu a dílčí souhrny byly následně spojeny do konečného výsledku.

Kontrola pomocí `np.allclose()` potvrdila, že zpracování po částech poskytlo stejný výsledek jako agregace celého DataFrame.

Chunks nesnižují množství dat, které je nutné ze CSV přečíst, ale omezují množství dat držených současně v operační paměti.